# HW5 - Latent Diffusion Or Latent Flow

Question: can we generate images by generating latents?

Sources:

- Latent Diffusion: https://arxiv.org/abs/2112.10752
- DDPM: https://arxiv.org/abs/2006.11239
- Flow Matching: https://arxiv.org/abs/2210.02747


In [ ]:
import math
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(7)
random.seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
COLORS = {
    "red": torch.tensor([1.0, 0.08, 0.08]),
    "green": torch.tensor([0.08, 0.80, 0.20]),
    "blue": torch.tensor([0.08, 0.25, 1.0]),
    "yellow": torch.tensor([1.0, 0.88, 0.08]),
}
SHAPES = ["circle", "square", "triangle"]


def draw_triangle(xx, yy, scale):
    # Upright triangle mask in normalized coordinates.
    top = yy > -scale
    left = yy < 2 * scale * (xx + scale)
    right = yy < -2 * scale * (xx - scale)
    return top & left & right & (yy < scale)


def make_shape_batch(batch=64, size=32, with_text=True):
    yy, xx = torch.meshgrid(
        torch.linspace(-1, 1, size),
        torch.linspace(-1, 1, size),
        indexing="ij",
    )
    images, captions = [], []
    for _ in range(batch):
        color_name = random.choice(list(COLORS))
        shape_name = random.choice(SHAPES)
        color = COLORS[color_name][:, None, None]
        scale = random.uniform(0.42, 0.72)
        img = torch.zeros(3, size, size)
        if shape_name == "circle":
            mask = xx.square() + yy.square() < scale**2
        elif shape_name == "square":
            mask = (xx.abs() < scale) & (yy.abs() < scale)
        else:
            mask = draw_triangle(xx, yy, scale)
        img[:, mask] = color
        images.append(img)
        captions.append(f"{color_name} {shape_name}")
    images = torch.stack(images)
    return (images, captions) if with_text else images


def show_images(images, titles=None, n=16, size=2.0):
    images = images.detach().cpu().clamp(0, 1)
    n = min(n, images.shape[0])
    cols = min(8, n)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size))
    axes = [axes] if n == 1 else axes.reshape(-1)
    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < n:
            ax.imshow(images[i].permute(1, 2, 0))
            if titles:
                ax.set_title(titles[i], fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
T = 64
betas = torch.linspace(1e-4, 0.035, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def q_sample(x0, t, eps=None):
    if eps is None:
        eps = torch.randn_like(x0)
    ab = alpha_bars[t].view(-1, *([1] * (x0.ndim - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, eps


@torch.no_grad()
def ddpm_sample(model, shape, cond=None, guidance_scale=None, cond_fn=None):
    x = torch.randn(shape, device=device)
    for ti in reversed(range(T)):
        t = torch.full((shape[0],), ti, device=device, dtype=torch.long)
        if guidance_scale is None:
            pred_eps = model(x, t) if cond is None else model(x, t, cond)
        else:
            eps_uncond = cond_fn(x, t, None)
            eps_cond = cond_fn(x, t, cond)
            pred_eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
        beta, alpha, ab = betas[ti], alphas[ti], alpha_bars[ti]
        mean = (1 / alpha.sqrt()) * (x - beta / (1 - ab).sqrt() * pred_eps)
        x = mean if ti == 0 else mean + beta.sqrt() * torch.randn_like(x)
    return x


This notebook trains a tiny VAE and then a small DDPM in its latent space.


In [ ]:
class TinyAE(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.SiLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.SiLU(),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, latent_dim),
        )
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 64 * 8 * 8),
            nn.Unflatten(1, (64, 8, 8)),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.enc(x)

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        return self.decode(self.encode(x))


ae = TinyAE().to(device)
opt_ae = torch.optim.AdamW(ae.parameters(), lr=2e-3)
for step in range(700):
    x = make_shape_batch(64, with_text=False).to(device)
    recon = ae(x)
    loss = F.mse_loss(recon, x)
    opt_ae.zero_grad(); loss.backward(); opt_ae.step()
    if step % 200 == 0:
        print("ae", step, round(loss.item(), 4))


In [ ]:
class LatentDenoiser(nn.Module):
    def __init__(self, latent_dim=64, hidden=192):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, latent_dim),
        )

    def forward(self, z, t):
        t_scaled = t.float()[:, None] / (T - 1)
        return self.net(torch.cat([z, t_scaled], dim=1))


denoiser = LatentDenoiser().to(device)
opt = torch.optim.AdamW(denoiser.parameters(), lr=2e-3)

for step in range(1200):
    x = make_shape_batch(128, with_text=False).to(device)
    with torch.no_grad():
        z0 = ae.encode(x)
    t = torch.randint(0, T, (z0.shape[0],), device=device)
    zt, eps = q_sample(z0, t)
    pred = denoiser(zt, t)
    loss = F.mse_loss(pred, eps)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 300 == 0:
        print("latent ddpm", step, round(loss.item(), 4))


In [ ]:
with torch.no_grad():
    z = ddpm_sample(denoiser, (16, 64))
    imgs = ae.decode(z).cpu()
show_images(imgs, n=16)


## Interview Takeaway

Latent generation reduces the dimensionality the generator must model. The tradeoff is that all reconstruction errors and latent-space pathologies become generator errors later.
